# RAG avec LlamaIndex et FAISS
---
pip install llama-index llama-index-llms-ollama llama-index-embeddings-ollama llama-index-vector-stores-faiss faiss-cpu pypdf

LlamaIndex → фреймворк для RAG “из коробки”

FAISS → быстрый поиск по векторам в памяти (очень быстрый retrieval)

pypdf → автоматическое чтение PDF

Загрузка данных: documents = SimpleDirectoryReader("./data").load_data()

In [1]:
from llama_index.core import VectorStoreIndex, SimpleDirectoryReader, StorageContext
from llama_index.embeddings.ollama import OllamaEmbedding
from llama_index.llms.ollama import Ollama
from llama_index.vector_stores.faiss import FaissVectorStore

import faiss
import os

In [ ]:
# 1. LLM (Ollama) -----------------------
llm = Ollama(model="llama3.2:3b", temperature=0)

# 2. Embeddings -----------------------
embed_model = OllamaEmbedding(model_name="llama3.2:3b")

# ⚠️ dimension must match model
d = 3072
faiss_index = faiss.IndexFlatL2(d)

# 3. Vector store FAISS -----------------------
vector_store = FaissVectorStore(faiss_index=faiss_index)

storage_context = StorageContext.from_defaults(
    vector_store=vector_store
)

# 4. Load documents (txt + pdf + md) -----------------------
documents = SimpleDirectoryReader("./data").load_data()
print(len(documents))

# 5. Build index -----------------------
index = VectorStoreIndex.from_documents(
    documents,
    storage_context=storage_context,
    embed_model=embed_model
)

# 6. Query engine -----------------------
query_engine = index.as_query_engine()

5
Doc ID: 3c688a8e-6727-46d9-b1a9-8b64ffcfa94c
Text: %PDF-1.4 % 1 0 obj << /Type /Catalog /Version /1.4 /Pages 2 0 R
/StructTreeRoot 3 0 R /MarkInfo 4 0 R /Lang (fr-FR) /ViewerPreferences
5 0 R >> endobj 6 0 obj << /Title (2025 Dev IA - FORMATION AGILE)
/Creator (Canva) /Producer (Canva) /CreationDate
(D:20251215123417+00'00') /ModDate (D:20251215123406+00'00') /Keywords
(DAG7gt-Chrg,BAB_11LKOZE,0...


In [ ]:
# 7. Chat loop -----------------------
print("RAG bot ready. (type 'exit' to quit)")

while True:
    question = input("\nUser: ")

    if question.lower() in ["exit", "stop", "fin"]:
        break

    response = query_engine.query(question)

    print("\nBot:", response)